In [2]:
import os
import shutil
import numpy as np
import pandas as pd
from tensorflow.keras.preprocessing import image
from tensorflow.keras.models import load_model

MODEL_PATH = "../models/Model-RDC-4.5/model.h5"
DATASET_DIR = "../dataset_split/test"
OUTPUT_DIR = "../dataset_filtered2/test"

IMG_SIZE = (224, 224)
THRESHOLD = 0.7

# ==========================================
# SETUP
# ==========================================
os.makedirs(OUTPUT_DIR, exist_ok=True)

model = load_model(MODEL_PATH)
class_names = sorted(os.listdir(DATASET_DIR))

results = []

# ==========================================
# PREDIKSI SEMUA GAMBAR
# ==========================================
for cls in class_names:
    class_path = os.path.join(DATASET_DIR, cls)

    if not os.path.isdir(class_path):
        continue

    for file in os.listdir(class_path):
        img_path = os.path.join(class_path, file)

        try:
            img = image.load_img(img_path, target_size=IMG_SIZE)
            img_array = image.img_to_array(img) / 255.0
            img_array = np.expand_dims(img_array, axis=0)

            pred = model.predict(img_array, verbose=0)
            pred_class = np.argmax(pred)
            confidence = np.max(pred)

            pred_label = class_names[pred_class]

            # simpan ke list
            results.append({
                "file": file,
                "path": img_path,
                "true_label": cls,
                "pred_label": pred_label,
                "confidence": float(confidence)
            })

        except Exception as e:
            results.append({
                "file": file,
                "path": img_path,
                "true_label": cls,
                "pred_label": "ERROR",
                "confidence": 0.0
            })

# ==========================================
# PANDAS
# ==========================================
df = pd.DataFrame(results)

# ==========================================
# FILTER SALAH PREDIKSI SAJA
# ==========================================
wrong = df[df["true_label"] != df["pred_label"]]

# ==========================================
# PINDAHKAN / SALIN GAMBAR SALAH
# ==========================================
for _, row in wrong.iterrows():
    src = row["path"]
    cls = row["true_label"]

    target_dir = os.path.join(OUTPUT_DIR, cls)
    os.makedirs(target_dir, exist_ok=True)

    shutil.move(src, os.path.join(target_dir, os.path.basename(src)))
    # kalau mau pindah ganti:
    # shutil.move(src, os.path.join(target_dir, os.path.basename(src)))

# ==========================================
# RINGKASAN (TANPA SPAM PRINT)
# ==========================================
total = len(df)
total_wrong = len(wrong)
accuracy = (df["true_label"] == df["pred_label"]).mean()

print("\n=== RINGKASAN ===")
print(f"Total data       : {total}")
print(f"Salah prediksi   : {total_wrong}")
print(f"Akurasi          : {accuracy:.4f}")

print("\n=== CONTOH DATA SALAH ===")
print(wrong[["file", "true_label", "pred_label", "confidence"]].head(10))


=== RINGKASAN ===
Total data       : 252
Salah prediksi   : 18
Akurasi          : 0.9286

=== CONTOH DATA SALAH ===
                           file true_label   pred_label  confidence
5     local_16_aug_dark_185.jpg       alur        retak    0.544759
49   public_14_aug_dark_265.png       alur        retak    0.698685
66                 local_35.png     lubang         alur    0.873805
67                 local_55.png     lubang         alur    0.916672
89               public_342.jpg     lubang        retak    0.936471
91               public_346.jpg     lubang         alur    0.643591
111              public_467.jpg     lubang         alur    0.982666
135              public_101.jpg      retak  tidak_rusak    0.651269
164                public_2.jpg      retak       lubang    0.651681
165              public_204.jpg      retak  tidak_rusak    0.855757
